#### Installing the dependencies

In [1]:
pip install pandas numpy scikit-learn xgboost fastparquet jupyterlab

Note: you may need to restart the kernel to use updated packages.


#### Imports and Declarations

In [7]:
import pandas as pd
import time
from IPython.display import display

from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

DATASET_PATH = "talkingdata-adtracking-fraud-detection/train.csv" 
CHUNK_SIZE = 10000
# processed_chunk = -1

#### Activating the Data Stream

<p>This is to simulate and test a streaming environment. The following script ensures your machine won't crash from memory overload and accurately mimics the high-frequency data pipelines used by FinTech risk managers and AdTech real-time bidders.</p>

In [2]:
def simulate_data_stream_test(file_path, chunk_size):
    print(f"Initializing the data stream from \"{file_path}\"...")
    try:
        stream = pd.read_csv(file_path, chunksize=chunk_size)
        for batch_num, chunk in enumerate(stream, start=1):
            print(f"\n--- Received Batch {batch_num} ---")
            print(f"Data Shape: {chunk.shape}")
            print(f"Columns Detected: {list(chunk.columns)}")
            print("\nLive Preview:")
            print(chunk.head(10))
            print("\nStream test successful! Halting connection.")
            break
            
    except FileNotFoundError:
        print(f"Error: '{file_path}' not found.")

simulate_data_stream_test(DATASET_PATH, CHUNK_SIZE) 

Initializing the data stream from "talkingdata-adtracking-fraud-detection/train.csv"...

--- Received Batch 1 ---
Data Shape: (10000, 8)
Columns Detected: ['ip', 'app', 'device', 'os', 'channel', 'click_time', 'attributed_time', 'is_attributed']

Live Preview:
       ip  app  device  os  channel           click_time attributed_time  \
0   83230    3       1  13      379  2017-11-06 14:32:21             NaN   
1   17357    3       1  19      379  2017-11-06 14:33:34             NaN   
2   35810    3       1  13      379  2017-11-06 14:34:12             NaN   
3   45745   14       1  13      478  2017-11-06 14:34:52             NaN   
4  161007    3       1  13      379  2017-11-06 14:35:08             NaN   
5   18787    3       1  16      379  2017-11-06 14:36:26             NaN   
6  103022    3       1  23      379  2017-11-06 14:37:44             NaN   
7  114221    3       1  19      379  2017-11-06 14:37:59             NaN   
8  165970    3       1  13      379  2017-11-06 14:38:1


#### Transformer Function

In [3]:
def engineer_features(chunk):
    
    # 1. Extracting the time patterns
    chunk['click_time'] = pd.to_datetime(chunk['click_time'])
    chunk['hour'] = chunk['click_time'].dt.hour
    chunk['day'] = chunk['click_time'].dt.day
    
    # 2. Removing the columns that could lead to data leakage (artificial accuracy is prevented)
    if 'attributed_time' in chunk.columns:
        chunk = chunk.drop(columns=['attributed_time'])
        
    # 3. The intra-batch IP frequency is calculated
    ip_counts = chunk.groupby('ip').size().reset_index(name='ip_batch_clicks')
    chunk = chunk.merge(ip_counts, on='ip', how='left')
    
    # 4. Calculate intra-batch IP-App frequency
    ip_app_counts = chunk.groupby(['ip', 'app']).size().reset_index(name='ip_app_batch_clicks')
    chunk = chunk.merge(ip_app_counts, on=['ip', 'app'], how='left')
    
    # 5. Drop the original datetime column as models require numerical data
    chunk = chunk.drop(columns=['click_time'])
    
    return chunk


##### The first chunk is pulled, the transformation function is applied over it and thw formatted output is displayed

In [5]:
def simulate_data_stream(file_path, chunk_size):
    print(f"Initializing data stream from {file_path}...\n")
    
    try:
        # The chunksize parameter creates an iterator, acting like a live data feed
        stream = pd.read_csv(file_path, chunksize=chunk_size)
        
        for batch_num, chunk in enumerate(stream, start=1):
            print(f"--- Received Batch {batch_num} ---")
            print(f"Original Data Shape: {chunk.shape}")
            
            # Apply feature engineering to the incoming chunk
            processed_chunk = engineer_features(chunk.copy())
            
            print(f"Processed Data Shape: {processed_chunk.shape}")
            print(f"Engineered Columns: {list(processed_chunk.columns)}")
            print("\nLive Preview:")
            # Displaying a subset of columns to ensure it fits well in the console
            print(processed_chunk[['ip', 'app', 'is_attributed', 'hour', 'ip_batch_clicks', 'ip_app_batch_clicks']].head(10))
            
            # Halt after the first batch to verify the pipeline is stable
            print("\nFeature engineering test successful! Halting connection.")
            return processed_chunk
            
    except FileNotFoundError:
        print(f"Error: '{file_path}' not found.")
    


modified_chunk = simulate_data_stream(DATASET_PATH, CHUNK_SIZE)

Initializing data stream from talkingdata-adtracking-fraud-detection/train.csv...

--- Received Batch 1 ---
Original Data Shape: (10000, 8)
Processed Data Shape: (10000, 10)
Engineered Columns: ['ip', 'app', 'device', 'os', 'channel', 'is_attributed', 'hour', 'day', 'ip_batch_clicks', 'ip_app_batch_clicks']

Live Preview:
       ip  app  is_attributed  hour  ip_batch_clicks  ip_app_batch_clicks
0   83230    3              0    14                1                    1
1   17357    3              0    14                2                    2
2   35810    3              0    14                1                    1
3   45745   14              0    14               15                    3
4  161007    3              0    14                1                    1
5   18787    3              0    14                2                    2
6  103022    3              0    14                1                    1
7  114221    3              0    14                1                    1
8  165970 

In [6]:
X = modified_chunk.drop(columns=['is_attributed'])
y = modified_chunk['is_attributed']

In [8]:
num_pos = sum(y == 1) # Calculate class imbalance ratio dynamically
scale_weight = sum(y == 0) / num_pos if num_pos > 0 else 100

print(f"Calculated scale_pos_weight: {scale_weight:.2f}")

xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, scale_pos_weight=scale_weight, random_state=42, eval_metric='auc')

xgb_model.fit(X, y) # Training the model on our engineered chunk

# Generate predictions and probabilities
xgb_preds = xgb_model.predict(X)
xgb_probs = xgb_model.predict_proba(X)[:, 1]

print("\n--- XGBoost Training Successful ---")
try:
    print(f"ROC-AUC Score: {roc_auc_score(y, xgb_probs):.4f}")
except ValueError:
    print("ROC-AUC Score: N/A (Only one class present in this specific batch)")

Calculated scale_pos_weight: 433.78

--- XGBoost Training Successful ---
ROC-AUC Score: 0.9999
